# Fine-Tune the Exit Advisor

Supervised fine-tuning of an OpenAI base model to drive the **Exit Advisor** — the binary `end` / `dont_end` classifier consulted by the Main Agent whenever it proposes to end a conversation.

Pipeline:
1. Build labeled examples from `sms_conversations.json` — every recruiter turn becomes one example (`end` when the recruiter turn is labeled `end`, `dont_end` otherwise).
2. Random 80/20 train/test split (fixed seed, reproducible).
3. Write JSONL files in chat-completions format.
4. Upload training data and launch a Supervised Fine-Tuning (SFT) job on `gpt-4o-mini-2024-07-18`.
5. Poll until `succeeded`; print the `ft:…` model id.
6. Evaluate the fine-tuned model on the held-out test split (accuracy + confusion matrix) and compare against the base model on the same split.

After this notebook completes, paste the printed `ft:…` id into `.env` as `EXIT_ADVISOR_MODEL=ft:…` — `exit_advisor.py` reads that variable and uses the fine-tuned model in the live pipeline.

**Caveat:** the labeled set is small (~50 turns; ~40 train / ~10 test after the split). Single-prediction swings will move test accuracy by ~10 points, so read the headline number with appropriate skepticism.

## Setup

In [ ]:
import json
import random
import sys
import time
import pathlib
from datetime import datetime

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# Put the project root on sys.path so we can import the existing Exit Advisor
# prompt and the same conversation-formatting helper main.py uses at runtime —
# the training data must be shaped exactly like what the live pipeline feeds in.
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from app.modules.agents.exit_advisor.exit_advisor import EXIT_ADVISOR_PROMPT
from app.main import format_conversation_history

client = OpenAI()
print(f"Project root: {PROJECT_ROOT}")

## Build labeled examples

For each recruiter turn that carries a ground-truth label *and* has prior history, build one example:
- **input** = the prior turns rendered with `format_conversation_history` (same shape main.py feeds the Exit Advisor at runtime)
- **target** = the Exit Advisor's JSON output, with `action` mapped from the dataset label:
  - dataset `end` → `{"action": "end", ...}`
  - dataset `continue` / `schedule` → `{"action": "dont_end", ...}`

Openers (recruiter turns with no prior history) are skipped — there's nothing to classify.

The `reason` field in each target is a short, label-derived phrase. It exists only so the JSON shape matches what the live Exit Advisor returns; we never evaluate against it.

In [ ]:
DATA_PATH = PROJECT_ROOT / "tests" / "sms_conversations.json"
with open(DATA_PATH, encoding="utf-8") as f:
    conversations = json.load(f)

print(f"Conversations: {len(conversations)}")

In [ ]:
ROLE_MAP = {"candidate": "user", "recruiter": "assistant"}

REASON_FOR = {
    "end": "Conversation has reached a natural conclusion.",
    "dont_end": "Conversation is still active and should continue.",
}


def prior_as_messages(prior_turns):
    return [
        {"role": ROLE_MAP[t["speaker"]], "content": t["text"]}
        for t in prior_turns
    ]


examples = []
for conv in conversations:
    turns = conv["turns"]
    for i, turn in enumerate(turns):
        if turn["speaker"] != "recruiter" or turn["label"] is None:
            continue
        prior = turns[:i]
        if not prior:
            continue

        exit_label = "end" if turn["label"] == "end" else "dont_end"
        user_content = format_conversation_history(prior_as_messages(prior))
        assistant_content = json.dumps({
            "action": exit_label,
            "reason": REASON_FOR[exit_label],
        })

        examples.append({
            "messages": [
                {"role": "system", "content": EXIT_ADVISOR_PROMPT},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": assistant_content},
            ],
            "exit_label": exit_label,
        })

label_counts = pd.Series([e["exit_label"] for e in examples]).value_counts()
print(f"Total examples: {len(examples)}")
print("Label distribution:")
print(label_counts)

## Random 80/20 train/test split

Fixed seed so the split is reproducible across runs. The test split is held out from training and used only for evaluation.

In [ ]:
SEED = 42
TRAIN_FRAC = 0.8

rng = random.Random(SEED)
shuffled = examples.copy()
rng.shuffle(shuffled)

split_at = int(len(shuffled) * TRAIN_FRAC)
train_examples = shuffled[:split_at]
test_examples = shuffled[split_at:]

print(f"Train: {len(train_examples)} | Test: {len(test_examples)}")
print("Train label distribution:", pd.Series([e['exit_label'] for e in train_examples]).value_counts().to_dict())
print("Test  label distribution:", pd.Series([e['exit_label'] for e in test_examples]).value_counts().to_dict())

## Write JSONL files

OpenAI's SFT API expects one JSON object per line, each with a `messages` array. The training file gets uploaded; the test file stays local and is used directly for evaluation.

In [ ]:
TRAIN_PATH = PROJECT_ROOT / "tests" / "exit_advisor_train.jsonl"
TEST_PATH  = PROJECT_ROOT / "tests" / "exit_advisor_test.jsonl"

def write_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps({"messages": item["messages"]}) + "\n")

write_jsonl(TRAIN_PATH, train_examples)
write_jsonl(TEST_PATH,  test_examples)

print(f"Wrote {TRAIN_PATH} ({len(train_examples)} rows)")
print(f"Wrote {TEST_PATH}  ({len(test_examples)} rows)")

## Upload training data and launch the fine-tuning job

In [ ]:
training_file = client.files.create(
    file=open(TRAIN_PATH, "rb"),
    purpose="fine-tune",
)
print(f"Uploaded training file: {training_file.id}")

In [ ]:
BASE_MODEL = "gpt-4o-mini-2024-07-18"

ft_job = client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=training_file.id,
    method={"type": "supervised"},
)
print(f"Fine-tuning job created: {ft_job.id}")
print(f"Status: {ft_job.status}")

## Poll until the job finishes

Fine-tuning a few-dozen-example job on `gpt-4o-mini` typically completes in 5–15 minutes. The loop below checks every 30 seconds and stops when the job reaches a terminal state.

In [ ]:
TERMINAL = {"succeeded", "failed", "cancelled"}

while True:
    job = client.fine_tuning.jobs.retrieve(ft_job.id)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] status={job.status}")
    if job.status in TERMINAL:
        break
    time.sleep(30)

if job.status != "succeeded":
    raise RuntimeError(f"Fine-tuning job ended with status={job.status}")

fine_tuned_model = job.fine_tuned_model
print(f"\nFine-tuned model id: {fine_tuned_model}")
print("\nAdd this to your .env:")
print(f"EXIT_ADVISOR_MODEL={fine_tuned_model}")

## Evaluate on the held-out test split

Run both the base model and the fine-tuned model over the same test examples and compare. We extract the `action` field from each model's JSON output and compare it to the ground-truth `exit_label`.

In [ ]:
def predict(model_id, example):
    """Run a single test example through the given model and return its predicted action."""
    # The training shape is [system, user, assistant]; at inference we send everything
    # except the assistant turn and let the model produce it.
    messages = [m for m in example["messages"] if m["role"] != "assistant"]

    completion = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=0,
        response_format={"type": "json_object"},
    )
    raw = completion.choices[0].message.content

    try:
        return json.loads(raw).get("action", "invalid_json")
    except json.JSONDecodeError:
        return "invalid_json"


def evaluate(model_id, examples_to_score, label):
    y_true, y_pred = [], []
    for ex in tqdm(examples_to_score, desc=label):
        y_true.append(ex["exit_label"])
        y_pred.append(predict(model_id, ex))
    return y_true, y_pred

In [ ]:
base_true, base_pred = evaluate(BASE_MODEL, test_examples, label="base")
base_acc = accuracy_score(base_true, base_pred)
print(f"\nBase model accuracy: {base_acc:.3f}  ({sum(t == p for t, p in zip(base_true, base_pred))}/{len(base_true)})")

In [ ]:
ft_true, ft_pred = evaluate(fine_tuned_model, test_examples, label="fine-tuned")
ft_acc = accuracy_score(ft_true, ft_pred)
print(f"\nFine-tuned model accuracy: {ft_acc:.3f}  ({sum(t == p for t, p in zip(ft_true, ft_pred))}/{len(ft_true)})")
print(f"Delta vs base: {(ft_acc - base_acc) * 100:+.1f} percentage points")

## Confusion matrices

In [ ]:
LABELS = ["dont_end", "end"]

def show_cm(y_true, y_pred, title, ax):
    extras = sorted(set(y_pred) - set(LABELS))
    matrix_labels = LABELS + extras
    cm = confusion_matrix(y_true, y_pred, labels=matrix_labels)
    df = pd.DataFrame(cm, index=matrix_labels, columns=matrix_labels)
    sns.heatmap(df, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show_cm(base_true, base_pred, f"Base ({BASE_MODEL})\nacc={base_acc:.3f}", axes[0])
show_cm(ft_true,   ft_pred,   f"Fine-tuned\nacc={ft_acc:.3f}",            axes[1])
plt.tight_layout()
plt.show()

In [ ]:
print("=== Base model ===")
print(classification_report(base_true, base_pred, labels=LABELS, zero_division=0))
print("=== Fine-tuned model ===")
print(classification_report(ft_true, ft_pred, labels=LABELS, zero_division=0))

## Summary

If the fine-tuned model beats the base on the held-out split, paste the printed `ft:…` id into `.env` as

```
EXIT_ADVISOR_MODEL=ft:gpt-4o-mini-...
```

`app/modules/agents/exit_advisor/exit_advisor.py` reads that variable and uses the fine-tuned model whenever it is set; otherwise it falls back to the LLM passed in from the Main Agent. No other code changes are needed to put the fine-tuned model into the live pipeline.